In [32]:
import pandas as pd
import json
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
df = pd.read_table(r"final.csv", sep=";")
df = df.drop_duplicates()

In [33]:
df.drop(columns=['author', 'author_type', 'description', 'house_number', 'nearest_metro', 'phone', 'residential_complex', 'street', 'underground', 'url'], inplace=True)
columns = ['accommodation_type', 'deal_type', 'finish_type', 'heating_type', 'house_material_type', 'location', 'object_type']
for i in columns:
	print(df[i].value_counts())
df.drop(columns=['accommodation_type', 'deal_type', 'finish_type', 'heating_type', 'house_material_type', 'location', 'object_type'], inplace=True)


accommodation_type
flat    55152
Name: count, dtype: int64
deal_type
sale    55152
Name: count, dtype: int64
finish_type
-1                                                         40266
Без отделки                                                 5430
Без отделки, предчистовая                                   2505
Без отделки, предчистовая, чистовая                         1942
Чистовая                                                    1908
Без отделки, чистовая                                        964
Предчистовая                                                 458
Без отделки, предчистовая, черновая, чистовая                356
Предчистовая, чистовая                                       250
Без отделки, предчистовая, черновая                          235
-                                                            235
Без отделки, чистовая с мебелью                              106
Без отделки, чистовая, чистовая с мебелью                    104
Без отделки, предчистовая, чистова

In [34]:
df.drop(columns=["transport_score"], inplace=True)

df["district"] = df["district"].map(lambda x: x if pd.isna(x) else x.lower())

with open("county_district.json", 'r', encoding='utf-8') as file:
    districts = json.load(file)
new_districts = dict()
for i in districts:
    name = i['name'].lower()
    district_id = i['id']
    new_districts[name] = district_id
df['district'] = df['district'].replace(new_districts)
df['district'] = pd.to_numeric(df['district'], errors='coerce')
df = df.dropna(subset=['district'])
df['district'].value_counts()

df["kitchen_meters"] = df["kitchen_meters"].map(lambda x: x if x == -1 else float(x.split()[0].replace(',', '.')))
df["kitchen_meters"] = df["kitchen_meters"].astype("float64")
df["kitchen_meters"] = df["kitchen_meters"].map(lambda x: np.nan if x == -1.0 else x)

df["living_meters"] = df["living_meters"].map(lambda x: x if x == -1 else float(x.split()[0].replace(',', '.')))
df["living_meters"] = df["living_meters"].astype("float64")
df["living_meters"] = df["living_meters"].map(lambda x: np.nan if x == -1.0 else x)

df["rooms_count"] = df["rooms_count"].map(lambda x: np.nan if x == -1 else x)
df = df.dropna(subset=['rooms_count'])

df["total_meters"] = df["total_meters"].astype("float64")
df["total_meters"] = df["total_meters"].map(lambda x: np.nan if x == -1.0 else x)
df = df.dropna(subset=['total_meters'])

df["kitchen_meters"] = pd.to_numeric(df["kitchen_meters"], errors="coerce")

df['year_of_construction'] = pd.to_numeric(df['year_of_construction'], errors='coerce')
df["year_of_construction"] = df["year_of_construction"].replace(-1, np.nan)

for i in df.columns:
    print(df[i].value_counts())
    print("_____________________")
df = df.replace([-1, -1.0], np.nan)

df = pd.get_dummies(df, columns=["district"])

district
61.0     2683
51.0     1785
63.0     1775
97.0     1229
64.0     1199
         ... 
43.0       11
124.0      10
70.0        5
111.0       2
146.0       1
Name: count, Length: 126, dtype: int64
_____________________
floor
2     4080
3     3588
4     3312
5     3216
6     2368
      ... 
77       2
64       2
70       1
69       1
74       1
Name: count, Length: 79, dtype: int64
_____________________
floors_count
9     3698
12    2578
17    2410
5     2366
14    2047
      ... 
97       3
60       2
80       1
83       1
95       1
Name: count, Length: 81, dtype: int64
_____________________
kitchen_meters
10.0    2526
6.0     2036
8.0     1387
9.0     1386
15.0    1219
        ... 
63.3       1
52.5       1
54.2       1
84.9       1
50.4       1
Name: count, Length: 565, dtype: int64
_____________________
living_meters
20.0     907
19.0     717
18.0     716
30.0     480
45.0     449
        ... 
109.4      1
68.8       1
110.9      1
108.2      1
91.2       1
Name: count, Length

In [35]:
q1 = df['price'].quantile(0.01)
q95 = df['price'].quantile(0.95)

df = df[(df['price'] >= q1) & (df['price'] <= q95)]

y = df['price']
y = np.log1p(y)

df.drop(columns=['price'], inplace=True)
X = df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [36]:
scaler = RobustScaler()
cols_to_scale = ['kitchen_meters', 'living_meters', 'total_meters',
                 'year_of_construction', 'floor', 'floors_count',
                 'metro_distance', 'rooms_count']

X_train_scaled = X_train.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

X_test_scaled = X_test.copy()
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

imputer = SimpleImputer(strategy='median').set_output(transform="pandas")
X_train_imputed = imputer.fit_transform(X_train_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

In [37]:
linear_regressor = LinearRegression()
tree_regressor = DecisionTreeRegressor()
neighbors_regressor = KNeighborsRegressor(weights='distance', p=1)

In [38]:
linear_regressor.fit(X_train_imputed, y_train)
tree_regressor.fit(X_train_imputed, y_train)
neighbors_regressor.fit(X_train_imputed, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",1
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [39]:
linear_train_predict = linear_regressor.predict(X_train_imputed)
tree_train_predict = tree_regressor.predict(X_train_imputed)
neighbors_train_predict = neighbors_regressor.predict(X_train_imputed)

linear_mae_train = mean_absolute_error(y_train, linear_train_predict)
linear_mape_train = mean_absolute_percentage_error(y_train, linear_train_predict)
linear_r2_train = r2_score(y_train, linear_train_predict)
linear_rmse_train = root_mean_squared_error(y_train, linear_train_predict)

tree_mae_train = mean_absolute_error(y_train, tree_train_predict)
tree_mape_train = mean_absolute_percentage_error(y_train, tree_train_predict)
tree_r2_train = r2_score(y_train, tree_train_predict)
tree_rmse_train = root_mean_squared_error(y_train, tree_train_predict)

neighbors_mae_train = mean_absolute_error(y_train, neighbors_train_predict)
neighbors_mape_train = mean_absolute_percentage_error(y_train, neighbors_train_predict)
neighbors_r2_train = r2_score(y_train, neighbors_train_predict)
neighbors_rmse_train = root_mean_squared_error(y_train, neighbors_train_predict)

In [40]:
linear_test_predict = linear_regressor.predict(X_test_imputed)
tree_test_predict = tree_regressor.predict(X_test_imputed)
neighbors_test_predict = neighbors_regressor.predict(X_test_imputed)

In [41]:
linear_test_predict = np.expm1(linear_test_predict)
tree_test_predict = np.expm1(tree_test_predict)
neighbors_test_predict = np.expm1(neighbors_test_predict)
y_test = np.expm1(y_test)

In [42]:
linear_test = [mean_absolute_error(y_test, linear_test_predict),mean_absolute_percentage_error(y_test, linear_test_predict),
               r2_score(y_test, linear_test_predict), root_mean_squared_error(y_test, linear_test_predict)]

tree_test = [mean_absolute_error(y_test, tree_test_predict), mean_absolute_percentage_error(y_test, tree_test_predict),
             r2_score(y_test, tree_test_predict), root_mean_squared_error(y_test, tree_test_predict)]

neighbors_test = [mean_absolute_error(y_test, neighbors_test_predict),mean_absolute_percentage_error(y_test, neighbors_test_predict), r2_score(y_test, neighbors_test_predict), root_mean_squared_error(y_test, neighbors_test_predict)]

In [43]:
print('LinearRegression')
print("TRAIN")
print()
print("MAPE", linear_mape_train * 100)
print("MAE", linear_mae_train)
print("R", linear_r2_train)
print("RMSE", linear_rmse_train)
print()
print("TEST")
print("MAPE", linear_test[1] * 100)
print("MAE", linear_test[0])
print("R", linear_test[2])
print("RMSE", linear_test[3])
print("____________________________")
print("Tree")
print("TRAIN")
print()
print("MAPE", tree_mape_train * 100)
print("MAE", tree_mae_train)
print("R", tree_r2_train)
print("RMSE", tree_rmse_train)
print()
print("TEST")
print("MAPE", tree_test[1] * 100)
print("MAE", tree_test[0])
print("R", tree_test[2])
print("RMSE", tree_test[3])
print("____________________________")
print("KNN")
print("TRAIN")
print()
print("MAPE", neighbors_mape_train * 100)
print("MAE", neighbors_mae_train)
print("R", neighbors_r2_train)
print("RMSE", neighbors_rmse_train)
print()
print("TEST")
print("MAPE",neighbors_test[1] * 100)
print("MAE", neighbors_test[0])
print("R", neighbors_test[2])
print("RMSE", neighbors_test[3])
print("____________________________")


LinearRegression
TRAIN

MAPE 1.2093974161848196
MAE 0.2129257779316009
R 0.8400874741963017
RMSE 0.3010949691511059

TEST
MAPE 23.3036236552611
MAE 16387687.281525604
R -14.07027070406033
RMSE 200186992.63249895
____________________________
Tree
TRAIN

MAPE 0.00767087367571156
MAE 0.00133057470140113
R 0.9998811429250901
RMSE 0.00820870742252406

TEST
MAPE 8.945084810346282
MAE 4017008.697503002
R 0.9436411910264109
RMSE 12242108.713807108
____________________________
KNN
TRAIN

MAPE 0.007700569273213339
MAE 0.0013353979621687098
R 0.9998800140415065
RMSE 0.008247597726790347

TEST
MAPE 7.069290752377593
MAE 3467630.3798236856
R 0.946497607604413
RMSE 11927843.427494626
____________________________
